In [1]:
import pandas as pd

df = pd.read_csv("customer_shopping_behavior.csv")
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   str    
 3   Item Purchased          3900 non-null   str    
 4   Category                3900 non-null   str    
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   str    
 7   Size                    3900 non-null   str    
 8   Color                   3900 non-null   str    
 9   Season                  3900 non-null   str    
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   str    
 12  Shipping Type           3900 non-null   str    
 13  Discount Applied        3900 non-null   str    
 14  Promo Code Used         3900 non-null   str    
 15

In [3]:
df.describe()

,Customer ID,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [4]:
df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [10]:
df["Review Rating"] = df.groupby("Category")["Review Rating"].transform(lambda x: x.fillna(x.median()))
# groupby("Category")
# → splits data into groups (Electronics, Clothing, etc.)
# ["Review Rating"]
# → selects only that column inside each group
# transform()
# → sends each group’s Review Rating values into the function
# lambda() is an anonymous function

In [11]:
df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [15]:
# we will use snake_case for column names to improve readibility and run in Python and Sql without any error
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(" ","_")
df.columns
# we can see purchase_amount_(usd) in wrong format so we will change it's name
df = df.rename(columns = {"purchase_amount_(usd)" : "purchase_amount"})
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='str')

In [16]:
# Now we will do feature engineering and create some new column
labels = ["Young Adult","Adult","Middle-aged","Senior"]
df["age_group"] = pd.qcut(df["age"], q = 4, labels = labels)
# It divides the age column into 4 equal groups (quartiles) and assigns labels to each group.
df[["age", "age_group"]].head(10)

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle-aged


In [46]:
# create column purchase_frequency_days
# It is in text in our dataset so we will convert in number for better anlysis and showing in metrics
frequency_mapping = {
    "Fortnightly" : 14,
    "Weekly":  7,
    "Monthly" : 30,
    "Quarterly" : 90,
    "Bi-Weekly" : 14,
    "Annually" : 365,
    "Every 3 Months" : 90
}
df["purchase_frequency_days"] = df["frequency_of_purchases"].map(frequency_mapping)
# map function replace the text in numbers with corressponding no. of days

df[["purchase_frequency_days","frequency_of_purchases"]]


    
    

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
...,...,...
3895,7,Weekly
3896,14,Bi-Weekly
3897,90,Quarterly
3898,7,Weekly


In [37]:
df.columns
# Lets check discount_applied and Promo_code_used columns
# df[["discount_applied","promo_code_used"]]
# we don't need promo code because it's like an identifier, it doesn't add any anyaltical value  in it's raw form like You have mapping like: SAVE20 → 20% discount ,FESTIVE → festival campaign
# And both discount_applied and promo_code_used columns have same value
(df["discount_applied"] == df["promo_code_used"]).all()
# all() checks if ALL values are True.
# Both have same value so we will drop promo_code_used column
df = df.drop("promo_code_used", axis = 1)

KeyError: 'promo_code_used'

In [38]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='str')

In [41]:
# We have done EDA and Analysis in Python, Now we will load the dataset into database for structured data and deeper analysis
# Install Required Libraries
!pip install sqlalchemy pyodbc
# Import Libraries
import pandas as pd
from sqlalchemy import create_engine

# Create Connection to SQL Server
engine = create_engine("mssql+pyodbc://DESKTOP-7GTR354/Customer_Behaviour_Shopping_Analysis?driver=ODBC+Driver+17+for+SQL+Server")
# Load Data into SQL Table
df.to_sql("customer_behaviour",con = engine,if_exists = "replace",index = False)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


50